In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

import os
os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')

In [2]:
#retrieve data
cps14 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps14_with_nearest_college_imgfeat_with_college_indicator.parquet')
# cps13 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps13_with_nearest_college_imgfeat_with_college_indicator.parquet')
# cps12 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps12_with_nearest_college_imgfeat_with_college_indicator.parquet')

cps14.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place',
       ...
       'mosaiks_3992', 'mosaiks_3993', 'mosaiks_3994', 'mosaiks_3995',
       'mosaiks_3996', 'mosaiks_3997', 'mosaiks_3998', 'mosaiks_3999',
       'college_count_in_puma', 'has_college_in_puma'],
      dtype='object', length=4026)

In [3]:
cps14.head()

,person_num,log_wage,educ,exp,exp2,female,black,mv,age,birth_place,...,mosaiks_3992,mosaiks_3993,mosaiks_3994,mosaiks_3995,mosaiks_3996,mosaiks_3997,mosaiks_3998,mosaiks_3999,college_count_in_puma,has_college_in_puma
0,4,16.659324,0.0,20.0,400.0,1,0,7.0,26,1,...,1.316642,0.716429,1.452152,1.471457,0.116492,5.285796,0.007212,5.026694,2,1
1,3,16.979841,12.0,8.0,64.0,0,1,5.0,26,1,...,1.198570,0.456456,1.538894,1.547372,0.012397,5.090301,0.002686,4.947719,4,1
2,2,15.806328,12.0,0.0,0.0,0,1,5.0,18,1,...,1.322542,0.202977,1.509819,1.623803,0.007932,5.479442,0.023653,5.330654,3,1
3,1,16.964838,14.0,23.0,529.0,0,0,7.0,43,1,...,1.438792,0.710858,1.448475,1.489014,0.010588,5.751905,0.017108,5.456117,3,1
4,2,17.585665,18.0,17.0,289.0,1,0,7.0,41,1,...,1.438792,0.710858,1.448475,1.489014,0.010588,5.751905,0.017108,5.456117,3,1


In [6]:
print(cps14["has_college_in_puma"].isna().mean())

0.0


In [4]:
#dropna
cps14 = cps14.dropna(subset=["nearest_college_dist_km"]).copy()

In [7]:
#baseline 2SLS

formula1 = "log_wage ~ 1 + exp + exp2 + [educ ~ has_college_in_puma]"
base = IV2SLS.from_formula(formula=formula1, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

formula2 = "log_wage ~ 1 + exp + exp2 + female + black + [educ ~ has_college_in_puma]"
base2 = IV2SLS.from_formula(formula=formula2, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

print(base.summary)
print(base.first_stage)
print(base2.summary)
print(base2.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                     -0.8720
Estimator:                    IV-2SLS   Adj. R-squared:                -0.8720
No. Observations:              148770   F-statistic:                    2259.3
Date:                Thu, Jul 09 2026   P-value (F-stat)                0.0000
Time:                        18:06:56   Distribution:                  chi2(3)
Cov. Estimator:             clustered                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      11.232     2.3044     4.8740     0.0000      6.7152      15.748
exp           -0.0325     0.0478    -0.6805     0.49

In [8]:
# TSLS (baseline) with PUMA-fixed effect
formula3 = "log_wage ~ 1 + exp + exp2 + C(PUMA) + [educ ~ has_college_in_puma]"
base3 = IV2SLS.from_formula(formula=formula3, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

formula4 = "log_wage ~ 1 + exp + exp2 + female + black + C(PUMA) + [educ ~ has_college_in_puma]"
base4 = IV2SLS.from_formula(formula=formula4, data=cps14, weights=cps14["pums_weight"]).fit(
    cov_type="clustered",
    clusters=cps14["STATE_PUMA"]
)

print(base3.summary)
print(base3.first_stage)
print(base4.summary)
print(base4.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                     -11.022
Estimator:                    IV-2SLS   Adj. R-squared:                -11.116
No. Observations:              148770   F-statistic:                 2.806e+12
Date:                Thu, Jul 09 2026   P-value (F-stat)                0.0000
Time:                        18:16:20   Distribution:               chi2(1152)
Cov. Estimator:             clustered                                         
                                                                              
                                Parameter Estimates                                 
                  Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------------
Intercept            28.724     67.452     0.4258     0.6702     -103.48      160.93
exp                  0.3234 